In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. load QC neuron file
2. append columns for spikes, region, coordinates
3. concatenate across patients, and save

### variables

In [2]:
patient = 18

### 1. copy cluster-figs to new directory for easier QC
### 2. create corresponding preQC_df csv
note: clustID called unitID going forward


In [3]:
# 1. init new dir with only figs of clusters for easier QC
osort_CL_dir = f'../../results/2025{patient}/osort_mat/figs_clust'
os.makedirs(osort_CL_dir, exist_ok=True)

# 2. init df for pre-QC csv 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# go through existing fig dir
osort_figs_dir = f'../../results/2025{patient}/osort_mat/figs/5'
for fig_file in glob.glob(f'{osort_figs_dir}/*CL*.png'):

    # grab only individual CL figs
    if not 'ALL' in os.path.basename(fig_file):
        
        # 1. copy individual CL figs to new dir
        dest = os.path.join(osort_CL_dir, os.path.basename(fig_file))
        if os.path.exists(dest): print(f'File already exists, skipping')
        else:
            print(f'Copying {fig_file} to {dest}')
            os.system(f'cp {fig_file} {osort_CL_dir}/')
            
        # 2. append IDs for preQC_df csv
        chanIDs.append(int(os.path.basename(fig_file).split('_')[0][1:]))
        unitIDs.append(int(os.path.basename(fig_file).split('_')[2]))
print('1. copied CL figs to new dir')

# create the preQC_df and save
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if os.path.exists(preQC_df_path):
    print(f'File already exists, skipping: {preQC_df_path}')
else:
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)
print('2. created and saved preQC_df csv')
preQC_df

File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File already exists, skipping
File alrea

,chanID,unitID
0,193,421
1,193,467
2,194,179
3,196,308
4,197,203
...,...,...
65,221,3
66,222,4
67,223,2
68,223,3


### after having performed manual QC, load QC_df:

In [4]:
QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')

# create list of dropped unitIDS
dropped_unitIDs = QC_df[QC_df['keep'] != 1]['unitID'].astype(int).tolist()
dropped_unitIDs.extend([0, 1, 99999999])

# drop unitIDs
QC_df = QC_df[~QC_df['unitID'].isin(dropped_unitIDs)].copy().reset_index(drop=True)
QC_df

,chanID,unitID,keep,notes
0,193,421,1.0,FR <.5
1,193,467,1.0,NaN
2,200,833,1.0,autocorr
3,202,871,1.0,NaN
4,202,892,1.0,FR <.5
5,203,464,1.0,"FR<.5, autocorr"
6,204,701,1.0,NaN
7,204,708,1.0,NaN
8,205,590,1.0,NaN
9,206,612,1.0,NaN


### helpers

In [5]:
def getunitID2spikes(unitIDs, spikes):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID in dropped_unitIDs: continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### start creating neur/spikes df. First, add spike data from OSort mats.

In [6]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    # if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
assert len(neur_df) == len(QC_df), f'Length mismatch: neur_df has {len(neur_df)} rows, QC_df has {len(QC_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR
0,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531
1,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812
2,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072
3,213,2153,"[0.6681666666666668, 0.9681333333333334, 2.171...",3999,2.402830
4,213,2104,"[10.1952, 27.753466666666668, 47.0327333333333...",1163,0.703192
5,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241
8,216,1079,"[2.503, 13.184033333333334, 28.113600000000005...",450,0.270778
9,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857


### add region column by mapping channel -> region (label)

In [7]:
chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

# variable across patients
if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

labelMap = chanMap['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))
neur_df

,chanID,unitID,spikes,num_spikes,FR,region
0,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2
1,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2
2,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2
3,213,2153,"[0.6681666666666668, 0.9681333333333334, 2.171...",3999,2.402830,mRAHIP5
4,213,2104,"[10.1952, 27.753466666666668, 47.0327333333333...",1163,0.703192,mRAHIP5
5,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853,mROFC8
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,mRACC7
8,216,1079,"[2.503, 13.184033333333334, 28.113600000000005...",450,0.270778,mRAHIP8
9,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857,mRAHIP8


### add coordinates columns by mapping region->coords

In [8]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

at some point, figure out this code line by line

In [9]:
# load
electrodeInfo = sio.loadmat(f'../../results/2025{patient}/records/2025{patient}_DI_Electrodes.mat')
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
# final_neur_df.to_csv(f'../../results/2025{patient}/records/df_spikes.csv', index=False)
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays


### inspect & save

In [10]:
print('example neuron')
print(f'last 5 spikes (s): {final_neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {final_neur_df["spikes"].iloc[0][-5:]/60}')

final_neur_df

example neuron
last 5 spikes (s): [1654.1982     1654.27323333 1654.47123333 1654.54063333 1655.09963333]
last 5 spikes (min): [27.56997    27.57122056 27.57452056 27.57567722 27.58499389]


,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2,16.600004,-3.000006,-16.400003
1,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2,16.600004,-3.000006,-16.400003
2,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2,16.600004,-3.000006,-16.400003
3,213,2153,"[0.6681666666666668, 0.9681333333333334, 2.171...",3999,2.402830,mRAHIP5,17.800004,-3.000006,-16.400003
4,213,2104,"[10.1952, 27.753466666666668, 47.0327333333333...",1163,0.703192,mRAHIP5,17.800004,-3.000006,-16.400003
5,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853,mROFC8,3.000004,44.199992,-10.400004
6,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7,9.000004,27.399993,23.199994
7,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,mRACC7,9.000004,27.399993,23.199994
8,216,1079,"[2.503, 13.184033333333334, 28.113600000000005...",450,0.270778,mRAHIP8,19.000003,-3.000006,-16.400003
9,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857,mRAHIP8,19.000003,-3.000006,-16.400003


In [ ]:
parquet_path = f'../../results/2025{patient}/records/processed_data/df_spikes.parquet'
if os.path.exists(parquet_path): print(f'File already exists, skipping: {parquet_path}')
else: final_neur_df.to_parquet(parquet_path, index=False)

File already exists, skipping: ../../results/202518/records/processed_data/df_spikes.parquet
